In [ ]:
!pip -q install tree-sitter==0.23.2 tree-sitter-language-pack==0.7.3 networkx==3.3 openai==1.54.0 numpy pydantic

In [ ]:
import os
import re
import json
import numpy as np
from collections import Counter
import textwrap
import shutil
from tree_sitter_language_pack import get_parser
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional
from openai import OpenAI

import warnings
warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = "sk-REPLACE-ME"
SCOUT_MODEL = "gpt-5-mini"
STRATEGIST_MODEL = "gpt-5"
VALIDATOR_MODEL = "gpt-5"
EMBED_MODEL = "text-embedding-3-small"

# 1) Introduction

Hand an LLM a single 200-line file with a bug and it does fine. Hand it a 50,000-LOC
repository and ask *"find the bugs"* and it falls apart. Three reasons, all
fundamental:

1. **Context window is finite.** Even with 1M-token windows, you cannot stuff a real
   codebase in and expect anything but blurry pattern matching. The model attends to
   the wrong things.
2. **Hallucination.** Without grounded retrieval, the model invents function
   signatures, claims a guard exists when it doesn't, fabricates call chains.
3. **Bugs are non-local.** Real bugs span functions and files. The RepoAudit paper
   puts it precisely: *"a code repository can be conceptualized as an enormous graph,
   with nodes representing individual statements and edges capturing intricate
   relationships such as control-flow, data-flow, and other interdependencies."*
   Reasoning across this graph is what static analyzers have done for decades. LLMs
   without that structure are flying blind.

> **Don't ask the LLM to read the repo. Build a structured model of the repo, let the
> LLM zoom into the slices it needs, and verify everything it claims.**

This is the architecture behind every state-of-the-art system in 2025:

| System | Key contribution | Result |
|---|---|---|
| **AutoCodeRover** (ISSTA'24) | AST-based code-search APIs (`search_class`, `search_method`) as agent tools | 46.2% on SWE-bench Verified |
| **RepoGraph** (ICLR'25) | k-hop ego-graph retrieval action augmenting existing agents | +32.8% relative gain on SWE-bench |
| **RepoAudit** (ICML'25) | Multi-agent + per-function data-flow summaries + validator | 78.4% precision, 185 zero-days |
| **LLM4PFA** (2025) | LLM as constraint-extractor + SMT solver for path feasibility | 72–96% false-positive reduction |
| **Hound** (2025) | Relation-first KGs ("aspect graphs") + persistent hypothesis beliefs | +6× recall over baseline LLM |

**What we will build**

A miniature version of the above that, given a Python repo, finds **structural
smells, dead code, taint-flow vulnerabilities, and resource leaks**, then writes a
report. The same architecture generalizes to any language tree-sitter supports
(165+ grammars).

```
            ┌─────────────────────────────────────────────────────────┐
            │                   AUDIT ORCHESTRATOR                     │
            └─────────────────────────────────────────────────────────┘
                       │                │                │
                  ┌────▼────┐     ┌─────▼────┐     ┌─────▼────┐
                  │  Scout  │     │Strategist│     │ Finalizer│
                  │(gpt-mini│     │ (gpt-5)  │     │ (gpt-5)  │
                  │ explore)│     │(hypothes)│     │ (verify) │
                  └────┬────┘     └─────┬────┘     └─────┬────┘
                       │                │                │
                       └────────────────┼────────────────┘
                                        ▼
            ┌─────────────────────────────────────────────────────────┐
            │      TOOLS:  get_function_body  |  get_callers           │
            │              get_callees       |  find_path              │
            │              semantic_search   |  read_file_slice        │
            └────────┬────────────────────────────┬───────────────────┘
                     │                            │
            ┌────────▼──────────┐        ┌────────▼──────────┐
            │  Knowledge Graph  │        │  Embedding Index  │
            │  (tree-sitter)    │        │  (OpenAI)         │
            └───────────────────┘        └───────────────────┘
```


## 1.1 Synthetic Repository With Planted Bugs

To make every detector concrete, we plant *known* bugs in a small repo. Each bug type
exercises a different detector we'll build. You'll see each one caught explicitly
later.

| Bug type | Where | What's wrong |
|---|---|---|
| **Dead code** | `utils.legacy_normalize` | Function exists, nothing calls it |
| **Call cycle** | `processor` ↔ `validator` | Mutual recursion that can stack-overflow |
| **God class** | `MegaService` | 14 methods, no cohesion |
| **Resource leak** | `cache.load_blob` | File opened, never closed on error path |
| **Taint flow (SQL-i)** | `auth.find_user` | Untrusted input flows into raw SQL |
| **Null-deref risk** | `pipeline.run` | Optional return used without guard |
| **Long function** | `pipeline.run` | 60+ LOC, multiple responsibilities |


In [ ]:
REPO = "/tmp/buggy_repo"
if os.path.exists(REPO):
    shutil.rmtree(REPO)
os.makedirs(REPO, exist_ok=True)


# Helper to write a file with consistent indentation handling.
def write(path: str, content: str):
    full = os.path.join(REPO, path)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(textwrap.dedent(content).lstrip("\n"))


# utils.py — contains a dead function (legacy_normalize) and a used one.
write(
    "utils.py",
    """
    def normalize(text):
        return text.strip().lower()

    def legacy_normalize(text):
        # Nobody calls this anymore — dead code.
        return text.upper().strip()
    """,
)


# auth.py — SQL-injection via string concatenation. Classic taint flow:
# request.body (source) -> username (param) -> raw SQL string (sink).

write(
    "auth.py",
    """
    import sqlite3

    def get_request_body():
        # Stand-in for an HTTP framework; in real code this is untrusted.
        return {"username": "alice' OR 1=1 --"}

    def find_user(conn, username):
        # SINK: raw concat into SQL. Tainted input reaches here unsanitized.
        query = "SELECT * FROM users WHERE name = '" + username + "'"
        return conn.execute(query).fetchone()

    def login(conn):
        body = get_request_body()
        # SOURCE -> SINK with no validation between.
        return find_user(conn, body["username"])
    """,
)


# cache.py — resource leak: file opened, exception path does not close it.
write(
    "cache.py",
    """
    def load_blob(path):
        f = open(path, "rb")     # opened
        data = f.read()
        if len(data) == 0:
            raise ValueError("empty blob")   # leak: f never closed here
        f.close()
        return data
    """,
)


# processor.py and validator.py — mutual recursion (call cycle).
write(
    "processor.py",
    """
    from validator import revalidate

    def process(item, depth=0):
        if item.get("dirty"):
            return revalidate(item, depth + 1)
        return item
    """,
)


write(
    "validator.py",
    """
    from processor import process

    def revalidate(item, depth=0):
        item["dirty"] = True
        return process(item, depth + 1)   # mutual recursion -> stack overflow risk
    """,
)


# service.py — god class anti-pattern (many unrelated methods on one class).
write(
    "service.py",
    """
    class MegaService:
        def login(self): pass
        def logout(self): pass
        def send_email(self): pass
        def generate_report(self): pass
        def compute_tax(self): pass
        def render_invoice(self): pass
        def upload_file(self): pass
        def download_file(self): pass
        def encrypt(self): pass
        def decrypt(self): pass
        def ping(self): pass
        def healthcheck(self): pass
        def backup(self): pass
        def restore(self): pass
    """,
)


# pipeline.py — long function with a null-deref risk.
write(
    "pipeline.py",
    """
    from utils import normalize

    def fetch_user(uid):
        # May return None if user not found. Caller must guard.
        if uid < 0:
            return None
        return {"id": uid, "name": "ada", "email": "ada@example.com"}

    def run(uid):
        user = fetch_user(uid)
        # NO None-check — direct attribute access -> AttributeError at runtime.
        name = normalize(user["name"])
        email = user["email"].lower()
        log_line = name + " " + email
        parts = log_line.split(" ")
        first = parts[0]
        last = parts[-1] if len(parts) > 1 else ""
        joined = first + "-" + last
        result = {}
        for ch in joined:
            result[ch] = result.get(ch, 0) + 1
        ordered = sorted(result.items(), key=lambda x: -x[1])
        top = ordered[:3]
        encoded = []
        for k, v in top:
            encoded.append(f"{k}={v}")
        out = ",".join(encoded)
        if not out:
            out = "EMPTY"
        return out
    """,
)


# main.py — ties it together.
write(
    "main.py",
    """
    from pipeline import run
    from auth import login
    from cache import load_blob

    def entrypoint():
        return run(42)
    """,
)


print("Repo files:")
for r, _, fs in os.walk(REPO):
    for f in fs:
        print(" ", os.path.relpath(os.path.join(r, f), REPO))

# 2) The Code Knowledge Graph

* **The problem with files:** code organizes physically by file but logically by
*relationship*. A bug usually involves a function in `auth.py`, a sink in `db.py`,
and a config in `settings.py`. Reading any one file alone tells you nothing.

* **The problem with embeddings alone:** semantic similarity ≠ control flow. Two
functions that *do similar things* may never call each other; two that *do* call each
other may look textually nothing alike.

* **The insight:** model code as a graph whose **edges are the relationships an auditor
cares about** — calls, imports, contains, inherits, references, data-flows-to. Then
*every* downstream analysis (dead code, taint, blast radius, retrieval for the LLM)
becomes a graph query.

This is exactly the move Hound makes ("relation-first" graphs), AutoCodeRover makes
(its search APIs are graph queries), and RepoGraph makes (k-hop ego-graphs as
retrieval).

**The Key Innovation: Two-Pass Extraction**

A subtle but critical detail. Tree-sitter parses *one file at a time*; calls to
`foo()` in file A might refer to `foo` defined in file B. If you try to resolve calls
during the first parse, you'll fail for every cross-file reference.

**Solution: two passes.**

- **Pass 1 — Definitions.** Walk every file, record every function/class definition in
  a global symbol index `name → node_id`.
- **Pass 2 — References.** Walk every file again. Now when you see a `call` node, you
  can look up the callee in the index built in pass 1.

This is the trick used by `tree-sitter-graph`, codemap, and GraphCode. Naive
single-pass attempts produce a graph with no cross-file edges — which is to say, no
useful graph at all.

**How It Works**

```
file.py  ──▶ tree-sitter ──▶  AST  ──▶ walker ──▶  events
                                              │
                       ┌──────────────────────┴────────────────────────┐
                       │                      │                        │
              function_definition       class_definition           call_node
                       │                      │                        │
                       ▼                      ▼                        ▼
               add node + edge        add node + edge          add edge from
              (CONTAINS file→fn)     (CONTAINS file→cls)      enclosing fn
                                                                to callee
```

**Sample Input → Output**

For `cache.py` (written in the cell above), the `load_blob` function becomes:
```
Nodes:
  cache.py                    kind=file
  cache.py::load_blob         kind=function, line=2, loc=6

Edges:
  cache.py        --CONTAINS--> cache.py::load_blob
  cache.py::load_blob --CALLS--> <builtin: open>     # if we track builtins
  cache.py::load_blob --CALLS--> <method: read>
```

**Common Pitfalls**

- **Don't confuse the AST with the call graph.** The AST has every `(` in your code; the
  call graph has only function-call edges between definitions. Filter aggressively.
- **Method resolution is hard.** `obj.foo()` — which `foo`? Without type info you must
  resort to name-based heuristics or LSP integration. We use name-based here (works
  well in practice for the 90% case).
- **Watch for shadowing.** Two files can both define `process`. A real system needs
  fully-qualified names (`module.path::process`). We use file-prefixed IDs.


In [ ]:
PARSER = get_parser("python")


# Node kinds in our KG. We treat the KG as a *typed* property graph.
# Each node has: id (str), kind (str), and arbitrary attributes (file, line, etc.).
# Each edge has: src, dst, kind.
@dataclass
class KGNode:
    id: str
    # "file" | "class" | "function" | "import"
    kind: str
    name: str
    file: str
    line: int = 0
    loc: int = 0
    # raw source code of the entity
    source: str = ""


@dataclass
class KGEdge:
    src: str
    dst: str
    # "contains" | "calls" | "imports" | "inherits"
    kind: str

In [ ]:
class KnowledgeGraph:
    """
    Property graph backed by two dicts. We build everything by hand so the mechanics
    are visible. NetworkX would work but obscures what's happening on storage.
    """

    def __init__(self):
        self.nodes: dict[str, KGNode] = {}
        self.edges: list[KGEdge] = []

        # Adjacency caches for fast traversal.
        self._out: dict[str, list[KGEdge]] = {}
        self._in: dict[str, list[KGEdge]] = {}

        # Name -> node_id index for cross-file resolution.
        self.symbol_index: dict[str, list[str]] = {}

    def add_node(self, node: KGNode) -> None:
        self.nodes[node.id] = node
        if node.kind in ("function", "class"):
            self.symbol_index.setdefault(node.name, []).append(node.id)

    def add_edge(self, src: str, dst: str, kind: str) -> None:
        e = KGEdge(src, dst, kind)
        self.edges.append(e)
        self._out.setdefault(src, []).append(e)
        self._in.setdefault(dst, []).append(e)

    # Query API the agents and detectors use.
    def out_edges(self, node_id, kind=None):
        return [e for e in self._out.get(node_id, []) if kind is None or e.kind == kind]

    def in_edges(self, node_id, kind=None):
        return [e for e in self._in.get(node_id, []) if kind is None or e.kind == kind]

    def successors(self, node_id, kind=None):
        return [e.dst for e in self.out_edges(node_id, kind)]

    def predecessors(self, node_id, kind=None):
        return [e.src for e in self.in_edges(node_id, kind)]

    def resolve(self, name: str) -> list[str]:
        # Look up a bare name (e.g. "process") to candidate fully-qualified node IDs.
        return self.symbol_index.get(name, [])

    def summary(self) -> str:
        kinds = Counter(n.kind for n in self.nodes.values())
        edge_kinds = Counter(e.kind for e in self.edges)
        return f"KG: {len(self.nodes)} nodes {dict(kinds)} | {len(self.edges)} edges {dict(edge_kinds)}"

In [ ]:
def _text(src_bytes: bytes, node) -> str:
    # Helper: tree-sitter gives byte offsets, decode the slice as UTF-8.
    return src_bytes[node.start_byte : node.end_byte].decode("utf-8", errors="replace")


def _node_id(file_path: str, name: str) -> str:
    # Globally-unique ID format. The "::" delimiter is conventional (LSP, Sourcegraph).
    return f"{file_path}::{name}"


# PASS 1: register every function and class definition
def pass1_definitions(kg: KnowledgeGraph, file_path: str, src: bytes) -> None:
    tree = PARSER.parse(src)

    # Register the file node itself.
    kg.add_node(
        KGNode(
            id=file_path,
            kind="file",
            name=Path(file_path).name,
            file=file_path,
            source=src.decode("utf-8", errors="replace"),
        )
    )

    # Walker that tracks the enclosing definition so nested defs become CONTAINS edges.
    def walk(node, parent_id: str):
        if node.type in ("function_definition", "class_definition"):
            name_node = node.child_by_field_name("name")
            if name_node is not None:
                name = _text(src, name_node)
                nid = _node_id(file_path, name)
                kind = "class" if node.type == "class_definition" else "function"
                kg.add_node(
                    KGNode(
                        id=nid,
                        kind=kind,
                        name=name,
                        file=file_path,
                        line=node.start_point[0] + 1,
                        loc=node.end_point[0] - node.start_point[0] + 1,
                        source=_text(src, node),
                    )
                )
                kg.add_edge(parent_id, nid, "contains")

                # New parent for any nested defs inside.
                parent_id = nid

        for child in node.children:
            walk(child, parent_id)

    walk(tree.root_node, file_path)


# PASS 2: extract calls and imports. By now every definition is in kg.symbol_index
def pass2_references(kg: KnowledgeGraph, file_path: str, src: bytes) -> None:
    tree = PARSER.parse(src)

    # For each AST node we need to know which function/class lexically encloses it,
    # so we can attribute the call edge to the right caller.
    def enclosing_def_id(node) -> str:
        cur = node.parent
        while cur is not None:
            if cur.type in ("function_definition", "class_definition"):
                name_node = cur.child_by_field_name("name")
                if name_node is not None:
                    return _node_id(file_path, _text(src, name_node))
            cur = cur.parent

        # If no enclosing function, attribute to the file itself (module-level call).
        return file_path

    def walk(node):
        if node.type == "call":
            fn_node = node.child_by_field_name("function")
            if fn_node is not None:

                # Take the rightmost identifier of e.g. `module.Class.method`.
                callee_name = _text(src, fn_node).split(".")[-1].strip()
                caller_id = enclosing_def_id(node)

                # Add an edge to every candidate definition (ambiguity-tolerant).
                for callee_id in kg.resolve(callee_name):
                    if callee_id != caller_id:
                        kg.add_edge(caller_id, callee_id, "calls")

        elif node.type in ("import_statement", "import_from_statement"):
            # Record imports as edges from this file to symbolic module names.
            module_text = _text(src, node).strip()
            mod_id = f"<module>::{module_text[:80]}"
            if mod_id not in kg.nodes:
                kg.add_node(
                    KGNode(id=mod_id, kind="import", name=module_text, file=file_path)
                )
            kg.add_edge(file_path, mod_id, "imports")
        for child in node.children:
            walk(child)

    walk(tree.root_node)


def build_kg(repo_path: str) -> KnowledgeGraph:
    kg = KnowledgeGraph()
    py_files = sorted(Path(repo_path).rglob("*.py"))

    # Pass 1 first across ALL files — crucial for cross-file resolution.
    for f in py_files:
        src = f.read_bytes()
        pass1_definitions(kg, str(f), src)
    for f in py_files:
        src = f.read_bytes()
        pass2_references(kg, str(f), src)
    return kg


kg = build_kg(REPO)
print(kg.summary())
print()
print("Sample nodes:")
for nid, n in list(kg.nodes.items())[:6]:
    print(f"  {n.kind:10s}  {nid}")
print()
print("Sample call edges:")
for e in [e for e in kg.edges if e.kind == "calls"][:6]:
    print(f"  {e.src}  --calls-->  {e.dst}")

# 3) Structural Detectors — Cheap, Exact, No LLM Needed

A surprising amount of bug-finding value comes from **pure graph algorithms** — no LLM,
no embeddings, no probabilistic anything. The graph is precise; if a function has no
incoming `calls` edges, it is *definitionally* uncalled (modulo dynamic dispatch).

Why bother with these if LLMs are so powerful? Three reasons:

1. **Free precision.** Zero false positives by construction — a SCC is a SCC.
2. **Free recall on certain bug classes.** The LLM may miss dead code in a 50K-line
   repo; the graph cannot.
3. **They focus the LLM.** Hotspots and cycles tell the agent *where to look*. This is
   the same logic AutoCodeRover uses with SBFL: feed cheap signals to the agent to
   bias its search.

**The Detectors**

| Detector | Algorithm | Bug class caught |
|---|---|---|
| Dead code | Reachability from entry points | Unused functions/classes |
| Call cycles | Strongly connected components | Mutual recursion, stack overflow risk |
| God objects | Out-degree on CONTAINS | Anti-cohesion |
| Long functions | LOC threshold | Maintainability, hidden complexity |
| Blast radius | Reverse BFS on CALLS | Change-risk hotspots |
| Fan-in hotspots | In-degree on CALLS | Critical pathways needing scrutiny |

**Key Algorithm — Tarjan's SCC**

Why Tarjan's? Because mutual recursion in code is exactly a non-trivial SCC in the
call graph. Naive cycle detection (DFS with a visited set) finds *some* cycles but
misses overlapping ones. Tarjan's gives you every maximal SCC in O(V+E) — optimal.

We implement it from scratch (not `networkx.strongly_connected_components`) so the
mechanics are visible. The algorithm uses two pieces of state per node:

- `index[v]` — DFS discovery order
- `lowlink[v]` — smallest index reachable from v via the DFS tree + at most one back
  edge

When `lowlink[v] == index[v]`, v is the root of an SCC, and the SCC consists of
everything above v on the stack.


In [ ]:
def detect_dead_code(
    kg: KnowledgeGraph, entry_names=("main", "entrypoint")
) -> list[KGNode]:
    """
    Reachability from declared entry points over the CALLS edges.
    Everything unreachable AND not a dunder/test/entry is dead.
    """
    # Resolve entry-point names to node IDs.
    frontier = set()
    for name in entry_names:
        frontier |= set(kg.resolve(name))

    # Standard BFS.
    reachable = set(frontier)
    while frontier:
        nxt = set()
        for node_id in frontier:
            for succ in kg.successors(node_id, kind="calls"):
                if succ not in reachable:
                    reachable.add(succ)
                    nxt.add(succ)
        frontier = nxt

    dead = []
    for nid, node in kg.nodes.items():
        if node.kind != "function":
            continue
        if node.name.startswith("__") or node.name.startswith("test_"):
            continue
        if nid in reachable:
            continue
        dead.append(node)
    return dead


for d in detect_dead_code(kg):
    print(f"DEAD:  {d.name}  ({d.file}:{d.line})")

In [ ]:
def detect_call_cycles(kg: KnowledgeGraph) -> list[list[str]]:
    """
    Tarjan's SCC implementation, from scratch, restricted to CALLS edges.
    Returns SCCs of size >= 2 (mutual recursion) plus self-loops (single recursion).
    """
    # Build a simple adjacency-list view restricted to CALLS edges.
    adj: dict[str, list[str]] = {}
    for e in kg.edges:
        if e.kind == "calls":
            adj.setdefault(e.src, []).append(e.dst)
            adj.setdefault(e.dst, [])

    index_counter = [0]
    index: dict[str, int] = {}
    lowlink: dict[str, int] = {}
    on_stack: set[str] = set()
    stack: list[str] = []
    sccs: list[list[str]] = []

    # Iterative Tarjan to avoid Python recursion-limit problems on deep graphs.
    def strongconnect(start: str):
        work = [(start, iter(adj.get(start, [])))]
        index[start] = index_counter[0]
        lowlink[start] = index_counter[0]
        index_counter[0] += 1
        stack.append(start)
        on_stack.add(start)

        while work:
            v, it = work[-1]
            try:
                w = next(it)
            except StopIteration:
                # Finished exploring v. Possibly emit an SCC rooted at v.
                if lowlink[v] == index[v]:
                    comp = []
                    while True:
                        x = stack.pop()
                        on_stack.discard(x)
                        comp.append(x)
                        if x == v:
                            break
                    sccs.append(comp)
                work.pop()
                if work:
                    parent = work[-1][0]
                    lowlink[parent] = min(lowlink[parent], lowlink[v])
                continue

            if w not in index:
                index[w] = index_counter[0]
                lowlink[w] = index_counter[0]
                index_counter[0] += 1
                stack.append(w)
                on_stack.add(w)
                work.append((w, iter(adj.get(w, []))))
            elif w in on_stack:
                lowlink[v] = min(lowlink[v], index[w])

    for v in list(adj.keys()):
        if v not in index:
            strongconnect(v)

    # Keep only non-trivial SCCs (size>=2) or single nodes with self-loops.
    nontrivial = []
    for comp in sccs:
        if len(comp) >= 2:
            nontrivial.append(comp)
        elif len(comp) == 1 and comp[0] in adj.get(comp[0], []):
            nontrivial.append(comp)
    return nontrivial


for cycle in detect_call_cycles(kg):
    print(f"CYCLE:  {' <-> '.join(kg.nodes[n].name for n in cycle)}")

In [ ]:
def detect_god_objects(
    kg: KnowledgeGraph, method_threshold: int = 10
) -> list[tuple[KGNode, int]]:
    out = []
    for nid, node in kg.nodes.items():
        if node.kind != "class":
            continue
        # Count contained functions (methods).
        methods = [
            e
            for e in kg.out_edges(nid, kind="contains")
            if kg.nodes[e.dst].kind == "function"
        ]
        if len(methods) >= method_threshold:
            out.append((node, len(methods)))
    return out


def detect_long_functions(kg: KnowledgeGraph, loc_threshold: int = 30) -> list[KGNode]:
    return [
        n for n in kg.nodes.values() if n.kind == "function" and n.loc >= loc_threshold
    ]


def blast_radius(kg: KnowledgeGraph, node_id: str, max_depth: int = 4) -> set[str]:
    """
    Reverse-BFS over CALLS to find every transitive caller — i.e., everything that
    breaks if `node_id` breaks. This is the standard 'impact analysis' query that
    tools like axon and code-review-graph expose to the agent.
    """
    visited = {node_id}
    frontier = {node_id}
    for _ in range(max_depth):
        nxt = set()
        for n in frontier:
            for pred in kg.predecessors(n, kind="calls"):
                if pred not in visited:
                    visited.add(pred)
                    nxt.add(pred)
        frontier = nxt
        if not frontier:
            break
    return visited - {node_id}


for cls, count in detect_god_objects(kg):
    print(f"GOD CLASS:  {cls.name}  ({count} methods)")
for f in detect_long_functions(kg):
    print(f"LONG FN:    {f.name}  ({f.loc} LOC)")

# 4) Inter-Procedural Data-Flow (RepoAudit)

Most security bugs are **taint-flow** bugs: untrusted input (a *source*) reaches a
dangerous operation (a *sink*) without intervening sanitization. SQL injection, XSS,
command injection, path traversal — all the same pattern at the graph level:

```
SOURCE ──data─flow──▶ ... ──▶ SINK
(HTTP body)                  (raw SQL exec)
```

To find these, you need to track how *values* move, not just how *functions* call
each other. A taint analysis answers: *can a value originating at source S reach
sink K, possibly through many intermediate functions?*

**The Key Innovation — Per-Function Summaries (RepoAudit)**

Classical inter-procedural analysis explodes because every function call can
introduce a new context. Compositional dataflow analysis (Reps, Horwitz, Sagiv 1995)
solved this with **summaries**: for each function, compute *"if these parameters are
tainted, are these returns tainted?"* once, then reuse.

RepoAudit applied this idea with LLMs. Per function, ask the model:

> *Given that parameter `p` is tainted, which return values, attributes, or
> arguments-to-callees become tainted?*

This is cheap (one LLM call per function) and produces a compact summary that
composes across the call graph.

**How It Works**

```
                   per-function summaries
                   ───────────────────────
   find_user(conn, username)
       INPUT TAINT:  username
       OUTPUT TAINT: (returns the row, also reaches SQL sink)

   login(conn)
       INPUT TAINT:  (none — body comes from external get_request_body)
       INTERNAL:     body["username"] is tainted (source)
                     -> passed to find_user(.., tainted)
                     -> find_user is tainted-summary "username -> sink"
       CONCLUSION:   sink reached, no sanitizer.
```

For this notebook we implement a simplified taint propagation:

1. Mark functions matching source patterns (`get_request_body`, `request.*`, `input`)
2. Mark sinks (`execute`, `exec`, `eval`, `subprocess.*`)
3. Walk the call graph from sources forward; record reachable sinks.

In production (RepoAudit) you'd also use an LLM to determine **whether** intermediate
functions sanitize. We do that in Section 6 (the Validator).


In [ ]:
# Pattern catalogs. In production these come from a config like Semgrep's rule files
# or CodeQL's taint specs. For pedagogy we hard-code three classes.
SOURCE_PATTERNS = [
    r"get_request_body",
    r"^request\.",
    r"sys\.argv",
    r"^input$",
]

SINK_PATTERNS = {
    "sql_injection": [r"\.execute\b", r"\.executemany\b"],
    "command_injection": [r"\bos\.system\b", r"subprocess\.(call|run|Popen)"],
    "code_injection": [r"\beval\b", r"\bexec\b"],
}


def classify(callee_text: str) -> tuple[str, str | None]:
    """
    Returns (role, label) where role in {'source', 'sink', None}.
    """
    for pat in SOURCE_PATTERNS:
        if re.search(pat, callee_text):
            return ("source", "external_input")
    for label, pats in SINK_PATTERNS.items():
        for pat in pats:
            if re.search(pat, callee_text):
                return ("sink", label)
    return (None, None)

In [ ]:
def find_source_functions(kg: KnowledgeGraph) -> set[str]:
    """
    A function is a 'source' if any callee inside it matches a source pattern,
    OR if the function's own name matches a source pattern (e.g. get_request_body).
    """
    sources = set()
    for nid, node in kg.nodes.items():
        if node.kind != "function":
            continue
        role, _ = classify(node.name)
        if role == "source":
            sources.add(nid)
            continue
        # Or any internal call to a source-named function:
        for e in kg.out_edges(nid, kind="calls"):
            callee = kg.nodes.get(e.dst)
            if callee and classify(callee.name)[0] == "source":
                sources.add(nid)
                break
    return sources


def find_sink_sites(kg: KnowledgeGraph) -> list[tuple[str, str]]:
    """
    Returns list of (containing_function_id, sink_label) for every callsite that
    looks like a sink. We approximate by scanning each function's source text for
    sink patterns — simple and effective for the pedagogical synthetic repo.
    """
    sinks = []
    for nid, node in kg.nodes.items():
        if node.kind != "function":
            continue
        for label, pats in SINK_PATTERNS.items():
            for pat in pats:
                if re.search(pat, node.source):
                    sinks.append((nid, label))
                    break
    return sinks

In [ ]:
def find_taint_paths(kg: KnowledgeGraph, max_depth: int = 6) -> list[dict]:
    """
    For each (source function, sink callsite) pair, find a call path connecting them.
    A 'path' is a sequence of function node IDs along CALLS edges.

    This is BFS to keep paths short; in real systems you'd weight by AST distance or
    sanitizer-presence heuristics.
    """
    sources = find_source_functions(kg)
    sinks = find_sink_sites(kg)
    sink_ids = {sid for sid, _ in sinks}
    sink_label = {sid: lab for sid, lab in sinks}

    findings = []
    for src in sources:
        # BFS from src.
        parents = {src: None}
        frontier = [src]
        for _ in range(max_depth):
            nxt = []
            for n in frontier:
                for succ in kg.successors(n, kind="calls"):
                    if succ in parents:
                        continue
                    parents[succ] = n
                    nxt.append(succ)
                    if succ in sink_ids:
                        # Reconstruct path.
                        path, cur = [], succ
                        while cur is not None:
                            path.append(cur)
                            cur = parents[cur]
                        path.reverse()
                        findings.append(
                            {
                                "kind": "taint_flow",
                                "label": sink_label[succ],
                                "source": src,
                                "sink": succ,
                                "path": path,
                            }
                        )
            frontier = nxt
            if not frontier:
                break
    return findings


taint = find_taint_paths(kg)
for f in taint:
    chain = " -> ".join(kg.nodes[n].name for n in f["path"])
    print(f"TAINT ({f['label']}):  {chain}")

# 3) The Validator — How LLMs Crush the False-Positive Rate

A pure static taint tracker is *recall-favored*: it overreports. Many supposed
"taint flows" are infeasible because:

- A guard condition (`if not is_safe(x): return`) blocks the dangerous path
- An intermediate function sanitizes (`escape(x)`)
- The "source" isn't actually attacker-controlled in this codebase
- The "sink" pattern matches but the call is on a different object

**Industrial static analyzers report a fact known in industry but unwritten in
textbooks: 70-90% of their findings are false positives that engineers learn to
ignore — which means real bugs get ignored too.** This is the noise problem.

**The Key Innovation (LLM4PFA, RepoAudit)**

> *Use the LLM as a path-feasibility judge.* Give it the candidate bug path, the
> source code along the path, and ask: *"Can a real input drive this path? Is anything
> on this path a sanitizer?"*

LLM4PFA reports 72–96% false-positive reduction this way. RepoAudit's validator
hits 78% precision on real open-source bugs. The trick is **focused prompting**: don't
ask the LLM to find bugs in the whole file; give it exactly the path and ask one
crisp question.

**How It Works**

```
candidate finding ──┐
                    ▼
   ┌──────────────────────────────────────────┐
   │  build prompt:                           │
   │     - bug type and pattern               │
   │     - source code of every function on   │
   │       the path                           │
   │     - explicit list of questions         │
   └──────────┬───────────────────────────────┘
              │
              ▼
   ┌──────────────────────────────────────────┐
   │   gpt-5 (Validator)                      │
   │   answers: feasible? sanitized? real?    │
   │   + reasoning trace                      │
   └──────────┬───────────────────────────────┘
              │
       ┌──────┴──────┐
       ▼             ▼
   CONFIRMED      DISMISSED
   (keep finding) (drop it)
```

**Sample Input → Output**

For our SQL-injection finding:

**Input to validator:**
```
Bug type: sql_injection
Path: login -> find_user
Source code of login(): ...
Source code of find_user(): ...
Question: Is there sanitization between source and sink? Is the path feasible?
```

**Expected output:**
```
{
  "feasible": true,
  "sanitized": false,
  "confidence": 0.9,
  "explanation": "find_user builds the SQL query by string concatenation with the
                  username parameter, which flows directly from get_request_body()
                  via login() without escaping or parameterization."
}
```

In [ ]:
# Lazy-init the client so the structural sections still run without a key.
_client = None


def llm() -> OpenAI:
    global _client
    if _client is None:
        _client = OpenAI()
    return _client


def chat(
    model: str,
    system: str,
    user: str,
    *,
    json_mode: bool = False,
    tools=None,
    tool_choice="auto"
):
    """
    Single-shot chat helper. Returns either parsed JSON, raw text, or the full
    message object when tools are passed.
    """
    kwargs = dict(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    if tools is not None:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    resp = llm().chat.completions.create(**kwargs)
    msg = resp.choices[0].message
    if tools is not None:
        return msg
    if json_mode:
        try:
            return json.loads(msg.content)
        except json.JSONDecodeError:
            return {"_raw": msg.content, "_error": "json_decode_failed"}
    return msg.content

In [ ]:
VALIDATOR_SYSTEM = """
You are a senior security auditor. You receive a candidate
bug finding and the source code on the path between source and sink. Your job is to
decide, conservatively, whether this is a REAL defect.

Answer strictly in JSON with keys:
  feasible    (bool)   - Can a realistic input drive the source-to-sink path?
  sanitized   (bool)   - Is the tainted value sanitized anywhere on the path?
  is_bug      (bool)   - Final verdict: real defect to fix.
  confidence  (float)  - In [0, 1].
  explanation (string) - One or two sentences. Cite line numbers if useful.

Be skeptical. If the path is unreachable, say so. If there's any sanitizer, say so.
"""


def validate_finding(kg: KnowledgeGraph, finding: dict) -> dict:
    # Assemble the focused prompt — source code of every function on the path.
    path_src = []
    for nid in finding["path"]:
        node = kg.nodes[nid]
        path_src.append(f"# {node.file}:{node.line}  ({node.name})\n{node.source}")
    code_block = "\n\n".join(path_src)

    user = f"""
    Bug class: {finding['label']}
    Source function:   {kg.nodes[finding['source']].name}
    Sink function:     {kg.nodes[finding['sink']].name}
    Call path:         {' -> '.join(kg.nodes[n].name for n in finding['path'])}

    Source code of the functions on the path:
    ---
    {code_block}
    ---

    Is this a real {finding['label']} defect?
    """

    verdict = chat(VALIDATOR_MODEL, VALIDATOR_SYSTEM, user, json_mode=True)
    return {**finding, "verdict": verdict}

# 6) Semantic Retrieval

The KG is excellent for *structural* questions ("who calls foo?", "what's the path
from A to B?") but bad at *semantic* questions ("where is the rate-limiting logic?",
"which functions handle payments?"). Names lie; comments lie too, but together with
the code they form a signal you can embed.

This is why every state-of-the-art repo agent has a **dual index**: graph + embeddings.
Hound, RepoGraph, CodexGraph, Codebase-Memory all do this. The agent uses the graph
to navigate *known* relationships and the embeddings to navigate *concepts*.

**The Key Innovation — AST-Aware Chunking**

**Don't chunk by lines.** A naive 500-line chunker splits in the middle of functions,
producing chunks whose embeddings are noise. The CAST paper (Code-Aware Structural
chunking via AST) made the obvious-in-hindsight observation: chunk by AST boundaries.
Every chunk is a whole function or class. Tools like Aider, Cursor, Continue all
chunk this way now.

The bonus: when retrieval surfaces a hit, you already know its `node_id` in your KG.
The two indices share IDs.

**How It Works**

```
   function source ──▶ openai.embeddings ──▶ vector ──▶ index[node_id]

   query  ────────────▶ openai.embeddings ──▶ vector
                                                │
                                                ▼
                                       cosine vs index
                                                │
                                                ▼
                                        top-k node_ids
                                                │
                                                ▼
                                   pull from KG, return source
```

**Common Pitfalls**

- **Chunking too granular.** A 3-line helper has too little context — its embedding
  is dominated by its name. Skip very short functions or merge with parent class.
- **Code vs natural language embeddings.** OpenAI's `text-embedding-3-small` handles
  both. For best results on code-specific tasks, consider Voyage's `voyage-code-3` or
  Salesforce's `SFR-Embedding-Code-v2`.
- **Ignoring the docstring.** Embed `def signature + docstring + body` for best
  semantic recall.


In [ ]:
def embed_texts(texts: list[str]) -> np.ndarray:
    """
    Batched embedding call. Returns (num_samples, embed_dim) float array.
    """
    # OpenAI accepts batched input; chunk to ~96 to stay below rate limits.
    out = []
    for i in range(0, len(texts), 96):
        batch = texts[i : i + 96]
        resp = llm().embeddings.create(model=EMBED_MODEL, input=batch)
        out.extend([d.embedding for d in resp.data])
    # (num_samples, embed_dim)
    return np.array(out, dtype=np.float32)


class SemanticIndex:
    """
    Minimal vector store: per-node embedding + cosine search.
    Sharing node IDs with the KG is what lets the agent fuse retrieval modes.
    """

    def __init__(self):
        self.ids: list[str] = []
        # (num_chunks, embed_dim) once built.
        self.matrix: np.ndarray | None = None

    def build(self, kg: KnowledgeGraph, min_loc: int = 2):
        # Eligible chunks: functions and classes with non-trivial body.
        eligible = [
            (nid, n)
            for nid, n in kg.nodes.items()
            if n.kind in ("function", "class") and n.loc >= min_loc
        ]
        if not eligible:
            return
        # Build the text we'll embed — include name + file + body for grounding.
        texts = [f"# {n.file} :: {n.name}\n{n.source}" for _, n in eligible]
        self.ids = [nid for nid, _ in eligible]
        # (num_chunks, embed_dim)
        self.matrix = embed_texts(texts)
        # Normalize for cosine via dot product later.
        norms = np.linalg.norm(self.matrix, axis=1, keepdims=True) + 1e-9
        self.matrix = self.matrix / norms

    def search(self, query: str, k: int = 5) -> list[tuple[str, float]]:
        if self.matrix is None or not self.ids:
            return []
        # (1, embed_dim)
        q = embed_texts([query])
        q = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-9)
        # (num_chunks,) — cosine similarities
        scores = (self.matrix @ q.T).squeeze(-1)
        order = np.argsort(-scores)[:k]
        return [(self.ids[i], float(scores[i])) for i in order]

In [ ]:
# Build the index (requires API key — wrap in try for offline runs).
sem = SemanticIndex()
try:
    sem.build(kg)
    print("indexed", len(sem.ids), "chunks")
    print()
    for nid, s in sem.search("SQL query construction", k=3):
        print(f"  {s:.3f}  {kg.nodes[nid].name}  ({kg.nodes[nid].file})")
except Exception as exc:
    print(f"(skipped — set OPENAI_API_KEY: {exc})")

# 7) The Multi-Agent Audit Architecture

A single LLM trying to audit a repo gets confused: it has to decide simultaneously
*what to investigate*, *how to investigate it*, and *whether the result is real*.
Specialization helps for the same reasons it helps in human teams.

**The Hound pattern (arXiv 2510.09633):**

- **Scout** — a cheap, fast model that *explores* the graph. Its job is breadth: hop
  through callers, callees, related semantic chunks, accumulate observations.
- **Strategist** — a strong model that *forms hypotheses*. Given the Scout's
  observations, it asks "what could go wrong here?" — generates concrete vulnerability
  hypotheses with evidence pointers.
- **Finalizer** — a strong model that *verifies*. For each hypothesis, it reads the
  code carefully, decides confirmed/rejected, writes the report.

The genius is the **persistent belief system**: hypotheses persist across iterations,
their confidence updates with new evidence, and the planning policy weighs *coverage*
(have we looked at everything?) against *intuition* (does this lead seem hot?).

**The Key Innovation — Tools That Wrap the Graph**

The Scout doesn't see the whole repo. It sees a small set of **tools** that wrap KG
queries. Each tool returns a few hundred tokens, not megabytes. AutoCodeRover's
contribution was showing this works: `search_method_in_class`, `search_code_around`,
etc. The agent navigates by composing tool calls.

We implement tools for:

| Tool | What it returns | Wraps |
|---|---|---|
| `get_function_body(name)` | Source code of the function | KG.nodes |
| `get_callers(name)` | Names of functions that call it | KG.in_edges |
| `get_callees(name)` | Names of functions it calls | KG.out_edges |
| `find_path(src, dst)` | Call chain between two functions | BFS on KG |
| `semantic_search(query)` | Top-k similar functions | SemanticIndex |
| `list_findings()` | Structural detector results | sections 4-5 |

**How It Works (ReAct loop)**

```
   ┌────────────────────────────────────────────────────────────┐
   │  while not done:                                            │
   │    1) Strategist proposes hypothesis OR refinement to test  │
   │    2) Scout executes tool calls to gather evidence          │
   │    3) Strategist updates hypothesis confidence              │
   │    4) If confidence > threshold: send to Finalizer          │
   │    5) Finalizer reads carefully, confirms or rejects        │
   │    6) Confirmed hypothesis → report                         │
   └────────────────────────────────────────────────────────────┘
```

**Sample Input → Output**

```
Strategist (turn 1): "find_user uses string concatenation in SQL. Hypothesis:
                     SQL injection if username is attacker-controlled."

Scout: get_callers("find_user") → ["login"]
       get_function_body("login") → "...body = get_request_body()..."

Strategist (turn 2): "login passes get_request_body()['username'] to find_user.
                     Get_request_body looks like an external source. Confidence
                     0.85. Send to Finalizer."

Finalizer:           "Confirmed. No sanitization. CWE-89."
```

** Common Pitfalls**

- **Token blowup.** Without budget caps, the Scout will recurse forever. Cap
  tool calls per hypothesis (e.g., 10).
- **Tool-call hallucination.** Validate every tool name and argument; reject and
  re-prompt on mismatch.
- **Hypothesis drift.** Each hypothesis should have a *target* (a node id). If the
  agent wanders away from its target, force it back.

In [1]:
def tool_get_function_body(kg: KnowledgeGraph, name: str) -> str:
    candidates = kg.resolve(name)
    if not candidates:
        return f"NOT_FOUND: {name}"
    out = []
    for nid in candidates:
        n = kg.nodes[nid]
        out.append(f"# {n.file}:{n.line}  ({n.name})\n{n.source}")
    return "\n\n".join(out)


def tool_get_callers(kg: KnowledgeGraph, name: str) -> list[str]:
    callers = set()
    for nid in kg.resolve(name):
        for pred in kg.predecessors(nid, kind="calls"):
            callers.add(kg.nodes[pred].name)
    return sorted(callers)


def tool_get_callees(kg: KnowledgeGraph, name: str) -> list[str]:
    callees = set()
    for nid in kg.resolve(name):
        for succ in kg.successors(nid, kind="calls"):
            callees.add(kg.nodes[succ].name)
    return sorted(callees)


def tool_find_path(
    kg: KnowledgeGraph, src: str, dst: str, max_depth: int = 8
) -> list[str]:
    src_ids = kg.resolve(src)
    dst_ids = set(kg.resolve(dst))
    if not src_ids or not dst_ids:
        return []
    # BFS from any src candidate to any dst candidate.
    for start in src_ids:
        parents = {start: None}
        frontier = [start]
        for _ in range(max_depth):
            nxt = []
            for n in frontier:
                for succ in kg.successors(n, kind="calls"):
                    if succ in parents:
                        continue
                    parents[succ] = n
                    if succ in dst_ids:
                        path, cur = [], succ
                        while cur is not None:
                            path.append(kg.nodes[cur].name)
                            cur = parents[cur]
                        return list(reversed(path))
                    nxt.append(succ)
            frontier = nxt
    return []


def tool_semantic_search(
    sem: SemanticIndex, kg: KnowledgeGraph, query: str, k: int = 5
) -> list[dict]:
    hits = sem.search(query, k=k)
    return [
        {"name": kg.nodes[nid].name, "file": kg.nodes[nid].file, "score": round(s, 3)}
        for nid, s in hits
    ]

NameError: name 'KnowledgeGraph' is not defined

In [ ]:
# OpenAI function-calling schema. Each tool becomes a callable from the agent's POV.
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_function_body",
            "description": "Return the source code of a function or class by name.",
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string"}},
                "required": ["name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_callers",
            "description": "Return the names of functions that call the given function.",
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string"}},
                "required": ["name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_callees",
            "description": "Return the names of functions called by the given function.",
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string"}},
                "required": ["name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "find_path",
            "description": "Find a call-graph path from one function to another, if any.",
            "parameters": {
                "type": "object",
                "properties": {
                    "src": {"type": "string"},
                    "dst": {"type": "string"},
                },
                "required": ["src", "dst"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "semantic_search",
            "description": "Return functions semantically similar to the query string.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "k": {"type": "integer", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "submit_finding",
            "description": (
                "Submit a confirmed bug finding. Use this only when you have "
                "high confidence and have gathered evidence."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "bug_class": {"type": "string"},
                    "location": {"type": "string"},
                    "evidence": {"type": "string"},
                    "severity": {
                        "type": "string",
                        "enum": ["low", "medium", "high", "critical"],
                    },
                    "confidence": {"type": "number"},
                },
                "required": [
                    "title",
                    "bug_class",
                    "location",
                    "evidence",
                    "severity",
                    "confidence",
                ],
            },
        },
    },
]


def dispatch_tool(name: str, args: dict, kg: KnowledgeGraph, sem: SemanticIndex) -> str:
    """
    Single dispatch point. Returns a JSON string so the LLM gets structured data.
    """
    try:
        if name == "get_function_body":
            return json.dumps({"body": tool_get_function_body(kg, args["name"])})
        if name == "get_callers":
            return json.dumps({"callers": tool_get_callers(kg, args["name"])})
        if name == "get_callees":
            return json.dumps({"callees": tool_get_callees(kg, args["name"])})
        if name == "find_path":
            return json.dumps({"path": tool_find_path(kg, args["src"], args["dst"])})
        if name == "semantic_search":
            return json.dumps(
                {
                    "results": tool_semantic_search(
                        sem, kg, args["query"], args.get("k", 5)
                    )
                }
            )
        if name == "submit_finding":
            return json.dumps({"ok": True, "stored": args})
        return json.dumps({"error": f"unknown tool {name}"})
    except Exception as e:
        return json.dumps({"error": str(e)})

In [ ]:
SCOUT_SYSTEM = """
You are the Scout in a code-auditing agent team. Your job is to
gather evidence about a specific hypothesis by composing tool calls. Each tool call
returns a small piece of the codebase. Be focused: investigate one hypothesis at a
time, and stop when you have enough evidence (typically 3-7 tool calls).

When you have enough evidence, call `submit_finding` with a clear title, location,
and the evidence chain. If after several calls the hypothesis is not supported,
submit nothing and say so plainly
"""


def run_scout(
    kg: KnowledgeGraph, sem: SemanticIndex, hypothesis: str, max_iters: int = 8
) -> dict:
    """
    Full ReAct loop. Each iteration the model decides which tool to call (or to
    submit a finding). We append tool results to the message history.
    """
    messages = [
        {"role": "system", "content": SCOUT_SYSTEM},
        {"role": "user", "content": f"Investigate this hypothesis:\n\n{hypothesis}"},
    ]
    findings: list[dict] = []

    for step in range(max_iters):
        resp = llm().chat.completions.create(
            model=SCOUT_MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
        )
        msg = resp.choices[0].message
        # Always append assistant message — even if no tool_calls — so history is consistent.
        messages.append(
            {
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [tc.model_dump() for tc in (msg.tool_calls or [])],
            }
        )

        if not msg.tool_calls:
            # Scout finished without further calls.
            break

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            result = dispatch_tool(tc.function.name, args, kg, sem)
            if tc.function.name == "submit_finding":
                findings.append(args)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                }
            )

    return {"findings": findings, "transcript": messages}

In [ ]:
STRATEGIST_SYSTEM = """You are the Strategist in a code-auditing agent team. You
receive a structural summary of the codebase plus precomputed signals from cheaper
analyses (dead code, cycles, taint candidates). Your job is to propose specific,
testable hypotheses about bugs and limitations, ranked by suspected severity.

Output JSON: {"hypotheses": [{"title": ..., "rationale": ..., "target": ...}, ...]}
where 'target' is the name of the function/class to investigate. Keep to <=5 hypotheses
to focus the Scout's limited budget."""


def run_strategist(kg: KnowledgeGraph, structural_report: dict) -> list[dict]:
    user = (
        "Codebase summary:\n"
        f"  files: {sum(1 for n in kg.nodes.values() if n.kind=='file')}\n"
        f"  functions: {sum(1 for n in kg.nodes.values() if n.kind=='function')}\n"
        f"  classes: {sum(1 for n in kg.nodes.values() if n.kind=='class')}\n\n"
        "Structural signals (precomputed):\n"
        f"{json.dumps(structural_report, indent=2)}\n\n"
        "Propose specific bug/limitation hypotheses to investigate."
    )
    out = chat(STRATEGIST_MODEL, STRATEGIST_SYSTEM, user, json_mode=True)
    return out.get("hypotheses", [])

# 8) End-to-End: Auditing the Buggy Repo

We now wire everything we've built into one orchestrator.

```
        repo path
            │
            ▼
   ┌────────────────────┐
   │  build_kg (§3)      │
   └─────────┬──────────┘
             │
             ├──▶ structural detectors (§4)      ─┐
             │                                    │
             ├──▶ taint detector (§5)             ├──▶ structural_report
             │                                    │
             └──▶ semantic index (§7)             │
                                                  ▼
                                       ┌─────────────────────┐
                                       │  Strategist (§8.4)  │  hypotheses
                                       └─────────┬───────────┘
                                                 ▼
                                       ┌─────────────────────┐
                                       │     Scout (§8.5)    │  per hypothesis
                                       └─────────┬───────────┘
                                                 ▼
                                       ┌─────────────────────┐
                                       │   Validator (§6)    │  for taint findings
                                       └─────────┬───────────┘
                                                 ▼
                                            audit report
```

- **Cheap signals first.** Structural detectors run in milliseconds and produce
  precise findings (dead code, cycles).
- **Strategist sees the signals.** This lets it prioritize hypotheses: "long
  function with high fan-in is worth scouting more than a 3-line helper."
- **Validator gates expensive findings.** Taint paths are noisy; the validator
  filters before they reach the report.

This is the same shape RepoAudit and Hound use.
